In [1]:
import json

file_path = "/home/user/Downloads/dubizzle_uae_2026_08_04.json"


def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)  # 1 MB at a time

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# Test reading the file
count = 0

for record in stream_json_objects(file_path):
    count += 1

    if count <= 5:
        print(f"Record {count}:")
        print(record)
        print()

    if count % 100000 == 0:
        print(f"Processed {count:,} records")

print(f"\nTotal records: {count:,}")

Record 1:
{'id': '60b7bc73d4c8338e62f57585eb71a553', 'agent_name': '', 'amenities': '', 'bathrooms': '', 'bedrooms': '', 'broker': '', 'broker_display_name': '', 'category': 'rent', 'category_url': '/property-for-rent/home/', 'completion_status': '', 'currency': 'AED', 'date': '2026-08-25', 'ded_license_number': '', 'depth': '', 'description': 'Semi furnished,Single room with common kitchen and bathroom available for an executive or a couple.Rent 1300 plus water and electricity charges.Free Wifi.Near Sahara centre, very close to Dubai.', 'details': '', 'dtcm_licence': '', 'furnished': '', 'iteration_number': '2026_08_04', 'last_update': '2026-08-20', 'latitude': '25.303382', 'listed_by': '', 'location': 'UAE,Sharjah,Al Nahda (Sharjah)', 'longitude': '55.377082757109', 'number_of_photos': '3', 'package_type': 'standard', 'phone_number': '+971552187031', 'price': '1300', 'price_per': 'Monthly', 'property_type': 'apartment', 'published_at': '2026-08-20', 'reference_number': '', 'rera_perm

In [2]:
from collections import Counter, defaultdict
import re


# ============================================================
# QA CONFIGURATION
# ============================================================

VALUE_COUNT_FIELDS = [
    "price_per",
    "furnished",
    "category",
    "listed_by",
    "package_type",
    "sub_category_1"
]

MAX_EXAMPLES = 10


# ============================================================
# URL VALIDATION
# ============================================================

def is_valid_url(value):

    if not isinstance(value, str):
        return False

    value = value.strip()

    if not value:
        return False

    return bool(
        re.match(r"^https?://[^\s]+$", value, re.IGNORECASE)
    )


# ============================================================
# VARIABLES
# ============================================================

total_records = 0
all_columns = set()

# Field presence / null / empty
column_present_count = Counter()
column_null_count = Counter()
column_empty_count = Counter()

# Value counts
value_counts = {
    field: Counter()
    for field in VALUE_COUNT_FIELDS
}

# Invalid URL
invalid_url_count = 0
invalid_url_examples = []

# Broker comparison
broker_mismatch_count = 0
broker_mismatch_examples = []

# Broker uppercase
broker_not_capital_count = 0
broker_not_capital_examples = []

# Spaces
space_issue_count = Counter()
space_examples = defaultdict(list)


# ============================================================
# PROCESS FILE
# ============================================================

print("Starting QA...")
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    if not isinstance(record, dict):
        continue


    # ========================================================
    # COLUMNS / SHAPE
    # ========================================================

    all_columns.update(record.keys())

    for column in record:
        column_present_count[column] += 1


    # ========================================================
    # NULL / EMPTY VALUES
    # ========================================================

    for column, value in record.items():

        if value is None:

            column_null_count[column] += 1

        elif isinstance(value, str) and value.strip() == "":

            column_empty_count[column] += 1


    # ========================================================
    # VALUE COUNTS
    # ========================================================

    for field in VALUE_COUNT_FIELDS:

        value = record.get(field)

        if value is None:

            value = "<NULL>"

        elif isinstance(value, str):

            if value.strip() == "":
                value = "<EMPTY>"
            else:
                value = value.strip()

        else:

            value = str(value)

        value_counts[field][value] += 1


    # ========================================================
    # INVALID URL
    # ========================================================

    url = record.get("url")

    if not is_valid_url(url):

        invalid_url_count += 1

        if len(invalid_url_examples) < MAX_EXAMPLES:

            invalid_url_examples.append({
                "record_number": record_number,
                "id": record.get("id", ""),
                "url": url
            })


    # ========================================================
    # BROKER vs BROKER_DISPLAY_NAME
    # ========================================================

    broker = record.get("broker")
    broker_display_name = record.get(
        "broker_display_name"
    )

    broker_str = (
        ""
        if broker is None
        else str(broker).strip()
    )

    broker_display_str = (
        ""
        if broker_display_name is None
        else str(broker_display_name).strip()
    )

    if broker_str and broker_display_str:

        if broker_str != broker_display_str:

            broker_mismatch_count += 1

            if len(broker_mismatch_examples) < MAX_EXAMPLES:

                broker_mismatch_examples.append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "broker": broker,
                    "broker_display_name": broker_display_name
                })


    # ========================================================
    # BROKER CAPITALIZATION
    # ========================================================

    if broker_str:

        if broker_str != broker_str.upper():

            broker_not_capital_count += 1

            if len(broker_not_capital_examples) < MAX_EXAMPLES:

                broker_not_capital_examples.append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "broker": broker
                })


    # ========================================================
    # UNWANTED LEADING / TRAILING SPACES
    # ========================================================

    for field, value in record.items():

        if not isinstance(value, str):
            continue

        if value != value.strip():

            space_issue_count[field] += 1

            if len(space_examples[field]) < MAX_EXAMPLES:

                space_examples[field].append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "value": value
                })


    # ========================================================
    # PROGRESS
    # ========================================================

    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("                         QA RESULT")
print("=" * 80)


# ============================================================
# 1. SHAPE
# ============================================================

print()
print("-" * 80)
print("1. DATA SHAPE")
print("-" * 80)

print(f"Rows    : {total_records:,}")
print(f"Columns : {len(all_columns):,}")
print(
    f"Shape   : ({total_records:,}, {len(all_columns):,})"
)


# ============================================================
# 2. COMPLETELY EMPTY COLUMNS
# ============================================================

print()
print("-" * 80)
print("2. COMPLETELY EMPTY COLUMNS")
print("-" * 80)

completely_empty = []

for column in sorted(all_columns):

    present = column_present_count[column]
    nulls = column_null_count[column]
    empty = column_empty_count[column]

    non_empty = present - nulls - empty

    if non_empty == 0:

        completely_empty.append(column)


if completely_empty:

    for column in completely_empty:
        print(column)

    print()
    print(
        f"Total completely empty columns: "
        f"{len(completely_empty)}"
    )

else:

    print("No completely empty columns found.")


# ============================================================
# 3. MISSING / NULL / EMPTY VALUES
# ============================================================

print()
print("-" * 80)
print("3. MISSING / NULL / EMPTY VALUES PER COLUMN")
print("-" * 80)

print(
    f"{'Column':<35}"
    f"{'Missing':>12}"
    f"{'NULL':>12}"
    f"{'Empty':>12}"
)

print("-" * 71)

for column in sorted(all_columns):

    present = column_present_count[column]

    missing = total_records - present

    nulls = column_null_count[column]

    empty = column_empty_count[column]

    print(
        f"{column:<35}"
        f"{missing:>12,}"
        f"{nulls:>12,}"
        f"{empty:>12,}"
    )


# ============================================================
# 4. INVALID URL
# ============================================================

print()
print("-" * 80)
print("4. INVALID URL")
print("-" * 80)

print(
    f"Total invalid URLs: {invalid_url_count:,}"
)

print()
print("First 10 invalid URLs")
print("-" * 80)

if invalid_url_examples:

    for item in invalid_url_examples:

        print(
            f"Record Number : {item['record_number']}"
        )
        print(
            f"ID            : {item['id']}"
        )
        print(
            f"URL           : {item['url']}"
        )
        print("-" * 80)

else:

    print("No invalid URLs found.")


# ============================================================
# 5. BROKER vs BROKER_DISPLAY_NAME
# ============================================================

print()
print("-" * 80)
print("5. BROKER vs BROKER_DISPLAY_NAME")
print("-" * 80)

print(
    f"Total mismatches: {broker_mismatch_count:,}"
)

if broker_mismatch_examples:

    print()
    print("First 10 mismatches")
    print("-" * 80)

    for item in broker_mismatch_examples:

        print(
            f"Record Number       : {item['record_number']}"
        )
        print(
            f"ID                  : {item['id']}"
        )
        print(
            f"URL                 : {item['url']}"
        )
        print(
            f"broker              : {item['broker']}"
        )
        print(
            f"broker_display_name : {item['broker_display_name']}"
        )
        print("-" * 80)

else:

    print("No mismatches found.")


# ============================================================
# 6. BROKER CAPITALIZATION
# ============================================================

print()
print("-" * 80)
print("6. BROKER CAPITALIZATION")
print("-" * 80)

print(
    f"Broker values not in CAPITALS: "
    f"{broker_not_capital_count:,}"
)

if broker_not_capital_examples:

    print()
    print("First 10 examples")
    print("-" * 80)

    for item in broker_not_capital_examples:

        print(
            f"Record Number : {item['record_number']}"
        )
        print(
            f"ID            : {item['id']}"
        )
        print(
            f"URL           : {item['url']}"
        )
        print(
            f"broker        : {item['broker']}"
        )
        print("-" * 80)

else:

    print("All non-empty broker values are in CAPITALS.")


# ============================================================
# 7. VALUE COUNTS
# ============================================================

print()
print("-" * 80)
print("7. VALUE COUNTS")
print("-" * 80)

for field in VALUE_COUNT_FIELDS:

    print()
    print(f"### {field}")

    for value, count in value_counts[field].most_common():

        print(
            f"{str(value):<40} : {count:,}"
        )


# ============================================================
# 8. UNWANTED SPACES
# ============================================================

print()
print("-" * 80)
print("8. LEADING / TRAILING SPACES")
print("-" * 80)

total_space_issues = sum(
    space_issue_count.values()
)

print(
    f"Total values with leading/trailing spaces: "
    f"{total_space_issues:,}"
)

if space_issue_count:

    print()
    print("Issue count by column:")

    for field, count in sorted(
        space_issue_count.items(),
        key=lambda x: x[1],
        reverse=True
    ):

        print(
            f"{field:<35} : {count:,}"
        )

    print()
    print("First 10 examples per affected column")

    for field, examples in space_examples.items():

        if not examples:
            continue

        print()
        print(f"### {field}")
        print("-" * 80)

        for item in examples:

            print(
                f"Record Number : {item['record_number']}"
            )
            print(
                f"ID            : {item['id']}"
            )
            print(
                f"URL           : {item['url']}"
            )
            print(
                f"Value         : {repr(item['value'])}"
            )
            print()

else:

    print("No leading/trailing spaces found.")


# ============================================================
# DONE
# ============================================================

print()
print("=" * 80)
print("QA COMPLETED")
print("=" * 80)

Starting QA...

Processed: 100,000 records
Processed: 200,000 records
Processed: 300,000 records
Processed: 400,000 records
Processed: 500,000 records
Processed: 600,000 records

                         QA RESULT

--------------------------------------------------------------------------------
1. DATA SHAPE
--------------------------------------------------------------------------------
Rows    : 617,511
Columns : 42
Shape   : (617,511, 42)

--------------------------------------------------------------------------------
2. COMPLETELY EMPTY COLUMNS
--------------------------------------------------------------------------------
ded_license_number
dtcm_licence
listed_by
user_id

Total completely empty columns: 4

--------------------------------------------------------------------------------
3. MISSING / NULL / EMPTY VALUES PER COLUMN
--------------------------------------------------------------------------------
Column                                  Missing        NULL       Empty

In [3]:
from collections import Counter


# ============================================================
# FIELDS TO CHECK
# ============================================================

CHECK_FIELDS = [
    "currency",
    "date",
    "scraped_ts",
    "published_at",
    "iteration_number",
    "last_update"
]

MAX_EXAMPLES = 10


# ============================================================
# COUNTERS
# ============================================================

value_counts = {
    field: Counter()
    for field in CHECK_FIELDS
}

total_records = 0


# ============================================================
# READ FILE
# ============================================================

print("Checking values...")
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    if not isinstance(record, dict):
        continue

    for field in CHECK_FIELDS:

        value = record.get(field)

        # Treat missing/null/empty separately
        if value is None:

            value = "<NULL>"

        elif isinstance(value, str):

            value = value.strip()

            if value == "":
                value = "<EMPTY>"

        else:

            value = str(value)

        value_counts[field][value] += 1


    # Progress
    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("VALUE COUNTS + UNIQUENESS CHECK")
print("=" * 80)


for field in CHECK_FIELDS:

    counts = value_counts[field]

    total_values = sum(counts.values())

    unique_values = len(counts)

    duplicate_values = sum(
        1
        for count in counts.values()
        if count > 1
    )

    duplicate_records = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )


    # --------------------------------------------------------
    # FIELD HEADER
    # --------------------------------------------------------

    print()
    print("-" * 80)
    print(f"FIELD: {field}")
    print("-" * 80)

    print(
        f"Total records       : {total_values:,}"
    )

    print(
        f"Unique values       : {unique_values:,}"
    )

    print(
        f"Duplicate values    : {duplicate_values:,}"
    )

    print(
        f"Duplicate records   : {duplicate_records:,}"
    )


    # --------------------------------------------------------
    # UNIQUENESS
    # --------------------------------------------------------

    if duplicate_values == 0:

        print("Unique?             : YES")

    else:

        print("Unique?             : NO")


    # --------------------------------------------------------
    # VALUE COUNTS
    # --------------------------------------------------------

    print()
    print("Value counts:")

    for value, count in counts.most_common():

        print(
            f"  {str(value):<35} : {count:,}"
        )


    # --------------------------------------------------------
    # DUPLICATE VALUES
    # --------------------------------------------------------

    duplicates = [
        (value, count)
        for value, count in counts.items()
        if count > 1
    ]

    if duplicates:

        print()
        print("First 10 duplicate values:")

        for value, count in sorted(
            duplicates,
            key=lambda x: x[1],
            reverse=True
        )[:MAX_EXAMPLES]:

            print(
                f"  {str(value):<35} : "
                f"{count:,} occurrences"
            )


# ============================================================
# COMPLETED
# ============================================================

print()
print("=" * 80)
print("CHECK COMPLETED")
print("=" * 80)

Checking values...

Processed: 100,000 records
Processed: 200,000 records
Processed: 300,000 records
Processed: 400,000 records
Processed: 500,000 records
Processed: 600,000 records

VALUE COUNTS + UNIQUENESS CHECK

--------------------------------------------------------------------------------
FIELD: currency
--------------------------------------------------------------------------------
Total records       : 617,511
Unique values       : 1
Duplicate values    : 1
Duplicate records   : 617,510
Unique?             : NO

Value counts:
  AED                                 : 617,511

First 10 duplicate values:
  AED                                 : 617,511 occurrences

--------------------------------------------------------------------------------
FIELD: date
--------------------------------------------------------------------------------
Total records       : 617,511
Unique values       : 1
Duplicate values    : 1
Duplicate records   : 617,510
Unique?             : NO

Value counts:

In [4]:
import sqlite3
import hashlib
import json
import tempfile
import os


# ============================================================
# SETTINGS
# ============================================================

MAX_EXAMPLES = 10


# ============================================================
# TEMPORARY DATABASE
# ============================================================

temp_db = tempfile.NamedTemporaryFile(
    suffix=".db",
    delete=False
)

db_path = temp_db.name
temp_db.close()


conn = sqlite3.connect(db_path)

cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE records (
        record_hash TEXT PRIMARY KEY,
        count INTEGER
    )
""")

conn.commit()


# ============================================================
# VARIABLES
# ============================================================

total_records = 0
unique_records = 0
duplicate_records = 0

duplicate_examples = []


# ============================================================
# PROCESS JSON
# ============================================================

print("=" * 80)
print("CHECKING COMPLETE RECORD UNIQUENESS")
print("=" * 80)
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    # --------------------------------------------------------
    # Create a consistent representation of the entire record
    # --------------------------------------------------------

    record_string = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

    # --------------------------------------------------------
    # Create hash
    # --------------------------------------------------------

    record_hash = hashlib.sha256(
        record_string.encode("utf-8")
    ).hexdigest()


    # --------------------------------------------------------
    # Check whether this complete record already exists
    # --------------------------------------------------------

    cursor.execute(
        """
        SELECT count
        FROM records
        WHERE record_hash = ?
        """,
        (record_hash,)
    )

    result = cursor.fetchone()


    if result is None:

        # Completely new record
        cursor.execute(
            """
            INSERT INTO records
            (record_hash, count)
            VALUES (?, 1)
            """,
            (record_hash,)
        )

        unique_records += 1

    else:

        # Duplicate complete record
        previous_count = result[0]

        cursor.execute(
            """
            UPDATE records
            SET count = count + 1
            WHERE record_hash = ?
            """,
            (record_hash,)
        )

        duplicate_records += 1

        # Keep only first 10 examples
        if len(duplicate_examples) < MAX_EXAMPLES:

            duplicate_examples.append({
                "record_number": record_number,
                "id": record.get("id", ""),
                "url": record.get("url", ""),
                "duplicate_occurrence": previous_count + 1
            })


    # --------------------------------------------------------
    # Commit periodically
    # --------------------------------------------------------

    if total_records % 10000 == 0:

        conn.commit()


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# Final commit
conn.commit()


# ============================================================
# FINAL RESULT
# ============================================================

cursor.execute(
    "SELECT COUNT(*) FROM records"
)

unique_hashes = cursor.fetchone()[0]


print()
print("=" * 80)
print("COMPLETE UNIQUENESS RESULT")
print("=" * 80)

print(
    f"Total records          : {total_records:,}"
)

print(
    f"Unique complete records: {unique_hashes:,}"
)

print(
    f"Duplicate records      : {duplicate_records:,}"
)


# ============================================================
# UNIQUE / NOT UNIQUE
# ============================================================

if duplicate_records == 0:

    print()
    print("RESULT: ALL RECORDS ARE COMPLETELY UNIQUE.")

else:

    print()
    print("RESULT: DUPLICATE COMPLETE RECORDS FOUND.")


# ============================================================
# DUPLICATE EXAMPLES
# ============================================================

print()
print("-" * 80)
print("FIRST 10 COMPLETE DUPLICATE RECORDS")
print("-" * 80)

if duplicate_examples:

    for item in duplicate_examples:

        print()
        print(
            f"Record Number       : "
            f"{item['record_number']}"
        )

        print(
            f"ID                  : "
            f"{item['id']}"
        )

        print(
            f"URL                 : "
            f"{item['url']}"
        )

        print(
            f"Occurrence          : "
            f"{item['duplicate_occurrence']}"
        )

else:

    print("No completely duplicate records found.")


# ============================================================
# CLOSE DATABASE
# ============================================================

conn.close()

# Delete temporary database
os.remove(db_path)

print()
print("=" * 80)
print("CHECK COMPLETED")
print("=" * 80)

CHECKING COMPLETE RECORD UNIQUENESS

Processed: 100,000 records
Processed: 200,000 records
Processed: 300,000 records
Processed: 400,000 records
Processed: 500,000 records
Processed: 600,000 records

COMPLETE UNIQUENESS RESULT
Total records          : 617,511
Unique complete records: 617,511
Duplicate records      : 0

RESULT: ALL RECORDS ARE COMPLETELY UNIQUE.

--------------------------------------------------------------------------------
FIRST 10 COMPLETE DUPLICATE RECORDS
--------------------------------------------------------------------------------
No completely duplicate records found.

CHECK COMPLETED
